# 01 — Grid Definition (Paris / EUBUCCO)

Defines a regular 150m grid over Paris and derives the **Y variable** (`zone_type`) from EUBUCCO building subtypes.

**Replaces:** NYC PLUTO with EUBUCCO open building dataset.

**Method:**
1. Stream EUBUCCO buildings for Paris (NUTS2: FR10) from S3
2. Filter to residential, commercial, industrial subtypes
3. Generate 150m x 150m regular grid covering Paris bounding box
4. Clip grid to convex hull of buildings (removes water/empty areas)
5. Assign buildings to grid cells
6. Derive zone_type by plurality of building subtype per cell

**Output columns:** `cell_id`, `cell_lat`, `cell_lon`, `zone_type`, `cell_building_count`

**Output file:** `csv/Paris/01_grid_definition.csv`

In [1]:
# ── Config ────────────────────────────────────────────
PARIS_CONFIG = "paris.json"

In [4]:
import pandas as pd
import numpy as np
import geopandas as gpd
import json
import os
import math
from scipy.spatial import ConvexHull
from shapely.geometry import Polygon, Point


In [7]:
storage_opts = {
    "anon": True,
    "client_kwargs": {"endpoint_url": "https://s3.eubucco.com"}
}
path = f"s3://eubucco/v0.2/buildings/parquet/nuts_id={NUTS_CODE}/{NUTS_CODE}.parquet"

import pyarrow.parquet as pq
import fsspec

fs = fsspec.filesystem(
    "s3",
    anon=True,
    client_kwargs={"endpoint_url": "https://s3.eubucco.com"}
)

with fs.open(path) as f:
    sample = pq.read_table(f).slice(0, 5).to_pandas()

print(sample.dtypes)
sample

id                                         str
region_id                                  str
city_id                                    str
type                                  category
subtype                               category
height                                  object
floors                                  object
construction_year                      float64
type_confidence                         object
subtype_confidence                      object
height_confidence_lower                 object
height_confidence_upper                 object
floors_confidence_lower                 object
floors_confidence_upper                 object
construction_year_confidence_lower     float64
construction_year_confidence_upper     float64
geometry_source                       category
type_source                           category
subtype_source                        category
height_source                         category
floors_source                         category
construction_

,id,region_id,city_id,type,subtype,height,floors,construction_year,type_confidence,subtype_confidence,...,construction_year_source,geometry_source_id,type_source_ids,subtype_source_ids,height_source_ids,floors_source_ids,construction_year_source_ids,subtype_raw,geometry,bbox
0,002fb5c5584b4c5e-0,FR101,FR75056,residential,detached,7.0,1.4,NaN,0.80,0.69,...,NaN,ile-de-france-latest_20354,None,None,None,None,None,NaN,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,"{'xmin': 3760481.6879135296, 'ymin': 2890341.4..."
1,004857a9736c4921-0,FR101,FR75056,residential,apartment,9.8,1.8,NaN,0.40,0.36,...,NaN,ile-de-france-latest_3273786,None,None,None,None,None,NaN,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,"{'xmin': 3762310.8704620004, 'ymin': 2889446.1..."
2,0048a12894544b88-0,FR101,FR75056,non-residential,others,6.0,1.3,NaN,None,None,...,NaN,ile-de-france-latest_3224089,[ile-de-france-latest_3224089],[ile-de-france-latest_3224089],None,None,None,roof,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,"{'xmin': 3764852.266809384, 'ymin': 2887605.70..."
3,007f31a707924d27-0,FR101,FR75056,residential,detached,3.6,1.0,NaN,0.78,0.78,...,NaN,ile-de-france-latest_3037416,None,None,None,None,None,NaN,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,"{'xmin': 3765289.446461845, 'ymin': 2886106.56..."
4,0085e5b30c9042c7-0,FR101,FR75056,residential,detached,4.6,1.1,NaN,0.70,0.60,...,NaN,ile-de-france-latest_967713,None,None,None,None,None,NaN,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,"{'xmin': 3759031.727595899, 'ymin': 2887221.74..."


In [9]:
with fs.open(path) as f:
    sample = pq.read_table(f, columns=["id", "type", "subtype", "floors"]).slice(0, 10).to_pandas()

sample

,id,type,subtype,floors
0,002fb5c5584b4c5e-0,residential,detached,1.4
1,004857a9736c4921-0,residential,apartment,1.8
2,0048a12894544b88-0,non-residential,others,1.3
3,007f31a707924d27-0,residential,detached,1.0
4,0085e5b30c9042c7-0,residential,detached,1.1
5,00dd23765e9c4757-0,non-residential,others,1.3
6,00f91b7701a74bac-0,residential,detached,1.2
7,012ee4f3c74344c0-0,residential,detached,1.2
8,0146ac1091294b67-0,residential,detached,1.0
9,016e315e2eac44a0-0,residential,detached,1.0


In [8]:
with fs.open(path) as f:
    types = pq.read_table(f, columns=["type"]).to_pandas()

print(types["type"].value_counts())

type
residential        2746022
non-residential     848973
Name: count, dtype: int64


In [6]:
with open(PARIS_CONFIG, encoding="utf-8") as f:
    config = json.load(f)


storage_opts = {
    "anon": True,
    "client_kwargs": {"endpoint_url": "https://s3.eubucco.com"}
}
path = f"s3://eubucco/v0.2/buildings/parquet/nuts_id={NUTS_CODE}/{NUTS_CODE}.parquet"
print(f"Streaming EUBUCCO for {NUTS_CODE}...")

gdf = gpd.read_parquet(path, storage_options=storage_opts)
gdf = gdf.to_crs("EPSG:4326")
gdf["latitude"]  = gdf.geometry.centroid.y
gdf["longitude"] = gdf.geometry.centroid.x

print(f"\nTotal buildings: {len(gdf):,}")
print(f"\nColumns:")
print(gdf.drop(columns='geometry').dtypes)
print(f"\nSample data:")
gdf.drop(columns='geometry').head(5)

Streaming EUBUCCO for FR10...


KeyboardInterrupt: 

In [2]:
import pandas as pd
import numpy as np
import geopandas as gpd
import json
import os
import math
from scipy.spatial import ConvexHull
from shapely.geometry import Polygon, Point

with open(PARIS_CONFIG, encoding="utf-8") as f:
    config = json.load(f)

NUTS_CODE     = config["nuts_code"]
CELL_SIZE_M   = config["grid_cell_size_m"]
MIN_BUILDINGS = config["min_buildings_per_cell"]
CSV_DIR       = config["csv_dir"]
TARGET_LABELS = config["target_labels"]

os.makedirs(CSV_DIR, exist_ok=True)

print(f"City:          {config['city']}")
print(f"NUTS code:     {NUTS_CODE}")
print(f"Grid cell:     {CELL_SIZE_M}m x {CELL_SIZE_M}m")
print(f"Min buildings: {MIN_BUILDINGS}")
print(f"Labels:        {TARGET_LABELS}")

City:          Paris, France
NUTS code:     FR10
Grid cell:     150m x 150m
Min buildings: 3
Labels:        ['residential', 'commercial', 'industrial']


In [3]:
# ── Stream EUBUCCO buildings from S3 ─────────────────
storage_opts = {
    "anon": True,
    "client_kwargs": {"endpoint_url": "https://s3.eubucco.com"}
}

path = f"s3://eubucco/v0.2/buildings/parquet/nuts_id={NUTS_CODE}/{NUTS_CODE}.parquet"
print(f"Streaming EUBUCCO from: {path}")

gdf = gpd.read_parquet(path, storage_options=storage_opts)
print(f"Loaded {len(gdf):,} buildings")

# Reproject to WGS84 and extract centroids
gdf = gdf.to_crs("EPSG:4326")
gdf["latitude"]  = gdf.geometry.centroid.y
gdf["longitude"] = gdf.geometry.centroid.x

print(f"\nSubtype distribution:")
print(gdf["subtype"].value_counts().to_string())

Streaming EUBUCCO from: s3://eubucco/v0.2/buildings/parquet/nuts_id=FR10/FR10.parquet
Loaded 3,594,995 buildings


C:\Users\Hani\AppData\Local\Temp\ipykernel_19676\2583268541.py:15: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["latitude"]  = gdf.geometry.centroid.y
C:\Users\Hani\AppData\Local\Temp\ipykernel_19676\2583268541.py:16: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["longitude"] = gdf.geometry.centroid.x



Subtype distribution:
subtype
detached         2264659
apartment         439545
others            416612
commercial        228183
industrial        118456
agricultural       71440
semi-detached      30589
public             14282
terraced           11229


In [ ]:
# ── Filter and label buildings ────────────────────────

# residential = top-level 'type' field (covers apartment, detached, etc.)
# commercial and industrial = 'subtype' field
conditions = (
    (gdf["type"] == "residential") |
    (gdf["subtype"] == "commercial") |
    (gdf["subtype"] == "industrial")
)
gdf_f = gdf[conditions].copy()
gdf_f["zone_cat"] = gdf_f.apply(
    lambda r: "residential" if r["type"] == "residential" else r["subtype"],
    axis=1
)
gdf_f = gdf_f.dropna(subset=["latitude", "longitude"]).copy()

print(f"Buildings after filter: {len(gdf_f):,}")
print(gdf_f["zone_cat"].value_counts().to_string())

In [ ]:
# ── Generate regular grid ─────────────────────────────
REF_LAT  = gdf_f["latitude"].mean()
LAT_STEP = CELL_SIZE_M / 111_000
LON_STEP = CELL_SIZE_M / (111_000 * math.cos(math.radians(REF_LAT)))

print(f"Reference latitude: {REF_LAT:.4f}")
print(f"Grid steps: lat={LAT_STEP:.6f}, lon={LON_STEP:.6f}")

BUFFER  = LAT_STEP
LAT_MIN = gdf_f["latitude"].min()  - BUFFER
LAT_MAX = gdf_f["latitude"].max()  + BUFFER
LON_MIN = gdf_f["longitude"].min() - BUFFER
LON_MAX = gdf_f["longitude"].max() + BUFFER

n_rows = int(math.ceil((LAT_MAX - LAT_MIN) / LAT_STEP))
n_cols = int(math.ceil((LON_MAX - LON_MIN) / LON_STEP))
print(f"Grid: {n_rows} rows x {n_cols} cols = {n_rows * n_cols:,} total cells")

# Convex hull to clip water/empty cells
coords = gdf_f[["longitude", "latitude"]].values
hull = ConvexHull(coords)
hull_polygon = Polygon(coords[hull.vertices])
print(f"Convex hull computed.")

In [ ]:
# ── Assign buildings to grid cells ───────────────────
gdf_f["grid_row"] = ((gdf_f["latitude"]  - LAT_MIN) / LAT_STEP).astype(int)
gdf_f["grid_col"] = ((gdf_f["longitude"] - LON_MIN) / LON_STEP).astype(int)
gdf_f["cell_id"]  = "r" + gdf_f["grid_row"].astype(str).str.zfill(4) + "_c" + gdf_f["grid_col"].astype(str).str.zfill(4)

PLURALITY_THRESHOLD = 0.40

cell_records = []
skipped_hull = 0
skipped_min  = 0

for (row, col), group in gdf_f.groupby(["grid_row", "grid_col"]):
    cell_lat = LAT_MIN + (row + 0.5) * LAT_STEP
    cell_lon = LON_MIN + (col + 0.5) * LON_STEP

    if not hull_polygon.contains(Point(cell_lon, cell_lat)):
        skipped_hull += 1
        continue

    if len(group) < MIN_BUILDINGS:
        skipped_min += 1
        continue

    # Plurality zone type
    counts = group["zone_cat"].value_counts()
    dominant       = counts.idxmax()
    dominant_ratio = counts[dominant] / len(group)
    zone_type = dominant if dominant_ratio >= PLURALITY_THRESHOLD else "mixed"

    cell_records.append({
        "cell_id":              f"r{row:04d}_c{col:04d}",
        "cell_lat":             round(cell_lat, 7),
        "cell_lon":             round(cell_lon, 7),
        "zone_type":            zone_type,
        "cell_building_count":  len(group),
    })

df_grid = pd.DataFrame(cell_records)
print(f"Skipped (outside hull): {skipped_hull}")
print(f"Skipped (< {MIN_BUILDINGS} buildings): {skipped_min}")
print(f"Final grid cells: {len(df_grid):,}")
print(df_grid["zone_type"].value_counts().to_string())

In [ ]:
# ── Save grid parameters to config ───────────────────
# So other notebooks can reconstruct the same grid
config["grid_params"] = {
    "lat_min": LAT_MIN, "lon_min": LON_MIN,
    "lat_step": LAT_STEP, "lon_step": LON_STEP,
    "ref_lat": REF_LAT
}
with open(PARIS_CONFIG, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4)
print("Grid params saved to paris.json")

# ── Save output ───────────────────────────────────────
output_path = f"{CSV_DIR}/01_grid_definition.csv"
df_grid.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}  ({len(df_grid)} rows x {df_grid.shape[1]} cols)")
df_grid.head(10)

In [ ]:
# ── Summary ───────────────────────────────────────────
print("Zone type breakdown:")
for zt in sorted(df_grid["zone_type"].unique()):
    subset = df_grid[df_grid["zone_type"] == zt]
    print(f"  {zt:<20s} {len(subset):>5d} cells  "
          f"(avg {subset['cell_building_count'].mean():.0f} buildings/cell)")
print(f"\nTotal cells: {len(df_grid):,}")
print(f"Lat range: {df_grid['cell_lat'].min():.4f} — {df_grid['cell_lat'].max():.4f}")
print(f"Lon range: {df_grid['cell_lon'].min():.4f} — {df_grid['cell_lon'].max():.4f}")